# Tick Compression Value Analysis

This investigation measures whether batching `xsend()` messages into a persistent zlib stream saves enough bandwidth to justify the additional protocol and implementation complexity. It uses deterministic, protocol-shaped fixtures rather than production traffic, so the results are comparative evidence rather than a traffic forecast.

The two primary outputs are:

1. **Average compression benefit by message type**, using marginal compressed output in shared stream contexts.
2. **Bandwidth with and without tick compression**, using the current frame threshold, framing overhead, 36 ticks/second, and a configurable active-player count.

## Measurement assumptions

The harness mirrors the current implementation:

- `xsend()` concatenates messages into one per-player tick buffer.
- `Server::compress_ticks()` uses a persistent zlib encoder with best compression and flushes after each tick batch.
- Every frame has a 2-byte length header; compressed frames set `0x8000`.
- Compression is selected when `raw_payload + 2 > 16`, not when compressed output is demonstrably smaller.
- The client keeps one continuous inflate stream, so fresh compression for every message is not the production behavior.
- `csend()` control packets are excluded because they bypass tick compression.

The notebook uses Python's standard library only. Its CPU timings are supporting evidence; the bandwidth conclusions are byte measurements and should be checked against Rust `flate2` if exact backend parity becomes important.

In [1]:
from dataclasses import dataclass
from collections import defaultdict
import itertools
import random
import statistics
import time
import zlib

TICKS_PER_SECOND = 36
ACTIVE_PLAYERS = 100
COMPRESSION_THRESHOLD = 16
BENCHMARK_TICKS = 100
WARMUP_TICKS = 20
SEED = 20260919

@dataclass(frozen=True)
class Message:
    family: str
    variant: str
    data: bytes

    @property
    def opcode(self):
        return self.data[0]

    def __repr__(self):
        return f"{self.family}/{self.variant} ({len(self.data)} B)"

def le16(value):
    return int(value).to_bytes(2, "little", signed=False)

def le32(value):
    return int(value).to_bytes(4, "little", signed=False)

def fixed(opcode, body=b""):
    return bytes([opcode]) + body

def fixed16(opcode, body=b""):
    return fixed(opcode, body[:15].ljust(15, b"\x00"))

def make(family, variant, data):
    return Message(family, variant, bytes(data))

def set_map(variant, delta, flags, index, seed):
    """Build SV_SETMAP using the server's flag-selected field widths."""
    data = bytearray([128 + delta, flags])
    if delta == 0:
        data.extend(le16(index))
    if flags & 1:
        data.extend(le16(seed))
    if flags & 2:
        data.extend(le32(seed * 17))
    if flags & 4:
        data.extend(le32(seed * 31))
    if flags & 8:
        data.extend(le16(seed * 3))
    if flags & 16:
        data.append(seed & 0xff)
    if flags & 32:
        data.extend(le32(seed * 7))
    if flags & 64:
        data.extend(bytes(((seed + offset) & 0xff) for offset in range(6)))
    if flags & 128:
        data.append((seed * 11) & 0xff)
    return make("SetMap", variant, data)

def build_fixtures(seed=SEED):
    rng = random.Random(seed)
    random_bytes = lambda count: bytes(rng.randrange(256) for _ in range(count))
    fixtures = []

    fixtures += [
        make("Tick", "idle", fixed(27, bytes([0]))),
        make("Tick", "late-cycle", fixed(27, bytes([35]))),
        make("Scroll", "right", bytes([30])),
        make("Scroll", "diagonal", bytes([35])),
        make("SetCharMode", "default", fixed(6, bytes([0]))),
        make("SetCharMode", "combat", fixed(6, bytes([7]))),
        make("SetCharDir", "north", fixed(71, bytes([0]))),
        make("SetCharDir", "south-east", fixed(71, bytes([5]))),
    ]

    name_a = b"Aster".ljust(15, b"\x00")
    name_b = bytes(range(15))
    fixtures += [
        make("SetCharName1", "common", fixed16(3, name_a)),
        make("SetCharName1", "high-entropy", fixed16(3, name_b)),
        make("SetCharAttrib", "zeros", fixed(7, bytes([0]) + bytes(12))),
        make("SetCharAttrib", "typical", fixed(7, bytes([2]) + b"\x14\x00\x18\x00\x1c\x00\x20\x00\x24\x00\x28\x00")),
        make("SetCharAttrib", "varied", fixed(7, bytes([4]) + random_bytes(12))),
        make("SetCharSkill", "typical", fixed(8, bytes([3]) + b"\x05\x00\x08\x00\x0d\x00\x12\x00\x19\x00\x21\x00")),
        make("SetCharSkill", "varied", fixed(8, bytes([9]) + random_bytes(12))),
    ]

    for opcode, family in [(12, "SetCharHp"), (13, "SetCharEndur"), (14, "SetCharMana")]:
        fixtures.append(make(family, "typical", fixed(opcode, b"\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00")))
        fixtures.append(make(family, "varied", fixed(opcode, random_bytes(12))))
    for opcode, family in [(20, "SetCharAHP"), (69, "SetCharAEnd"), (70, "SetCharAMana")]:
        fixtures.append(make(family, "low", fixed(opcode, le16(2))))
        fixtures.append(make(family, "high", fixed(opcode, le16(65530))))

    fixtures += [
        make("SetCharPts", "typical", fixed(21, le16(42) + le32(12000) + bytes([2]) + bytes(5))),
        make("SetCharGold", "typical", fixed(22, le32(9000) + le16(101) + le16(202) + bytes(4))),
        make("SetCharItem", "empty", fixed(23, bytes(8))),
        make("SetCharItem", "equipped", fixed(23, le16(7) + le16(320) + le32(123456))),
        make("SetCharWorn", "equipped", fixed(24, le16(4) + le16(900) + le32(654321))),
        make("SetCharSpell", "mixed", fixed(46, le16(1) + random_bytes(8))),
        make("SetCharObj", "typical", fixed(25, le16(16) + le16(700))),
    ]

    fixtures += [
        set_map("absolute-sprite", 0, 1, 1200, 21),
        set_map("absolute-full", 0, 255, 2048, 37),
        set_map("delta-sprite", 4, 1, 2052, 21),
        set_map("delta-full", 7, 255, 2059, 37),
        make("SetMap4", "one-light", fixed(66, le16(1200) + bytes([8]))),
        make("SetMap5", "three-light", fixed(67, le16(1200) + bytes([8, 0x87]))),
        make("SetMap6", "seven-light", fixed(68, le16(1200) + bytes([8, 0x87, 0x65, 0x43]))),
        make("SetMap3", "twenty-six-light", fixed(45, le16(1200) + bytes([8]) + bytes([0x87] * 13))),
        make("SetOrigin", "typical", fixed(44, le16(80) + le16(120))),
    ]

    fixtures += [
        make("SetCharTalents", "empty", fixed(75, bytes(25))),
        make("SetCharTalents", "varied", fixed(75, bytes((i * 7) & 0xff for i in range(25)))),
        make("SetCharRuneState", "inactive", fixed(78, bytes([0]) + le16(0))),
        make("SetCharRuneState", "active", fixed(78, bytes([3]) + le16(144))),
        make("SetCompletionData", "empty", fixed(77, bytes(41))),
        make("SetCompletionData", "varied", fixed(77, bytes((i * 13) & 0xff for i in range(41)))),
        make("SetWeather", "clear", fixed(76, bytes([0, 0]) + le16(0) + bytes(5))),
        make("SetWeather", "storm", fixed(76, bytes([4, 220]) + le16(720) + bytes([80, 90, 110, 255, 3]))),
    ]

    fixtures += [
        make("Log", "repetitive", fixed16(52, b"The same message")),
        make("Log", "random", fixed16(55, random_bytes(15))),
        make("Look", "repetitive", fixed16(29, bytes([35, 0] * 7 + [0]))),
        make("Look", "varied", fixed16(39, random_bytes(15))),
        make("SetTarget", "combat", fixed(42, le16(900) + le16(81) + le16(90) + bytes([3, 4, 5]) + bytes(4))),
        make("PlaySound", "typical", fixed(47, le16(12) + bytes([90, 128]) + bytes(9))),
        make("Pong", "typical", fixed16(74, le32(123456) + le32(7890))),
    ]
    return fixtures

fixtures = build_fixtures()
len(fixtures), len(set((message.family, message.variant) for message in fixtures))

(58, 58)

In [2]:
fixtures = [
    make("SetTarget", "combat", fixed(42, le16(900) + le16(81) + le16(90) + bytes([3, 4, 5]) + bytes(3)))
    if (message.family, message.variant) == ("SetTarget", "combat")
    else make("PlaySound", "typical", fixed(47, le16(12) + bytes([90, 128]) + bytes(8)))
    if (message.family, message.variant) == ("PlaySound", "typical")
    else message
    for message in fixtures
]

## Fixture validation

The parser below follows the fixed lengths and flag-selected `SetMap` widths in `core/src/server_commands.rs`. It is intentionally small and only needs to validate the messages generated by this notebook.

In [3]:
FIXED_LENGTHS = {
    3: 16, 6: 2, 7: 14, 8: 14, 12: 13, 13: 13, 14: 13,
    20: 3, 21: 13, 22: 13, 23: 9, 24: 9, 25: 5, 27: 2, 29: 16,
    30: 1, 31: 1, 32: 1, 33: 1, 35: 1, 36: 1, 37: 1, 38: 1,
    39: 16, 40: 16, 41: 16, 42: 13, 44: 5, 45: 17, 46: 11,
    47: 13, 50: 16, 52: 16, 53: 16, 54: 16, 55: 16, 66: 4,
    67: 5, 68: 7, 69: 3, 70: 3, 71: 2, 74: 16, 75: 26,
    76: 10, 77: 42, 78: 4,
}

def setmap_length(data, last_index):
    if len(data) < 2:
        raise ValueError("SetMap needs opcode and flags")
    opcode, flags = data[:2]
    offset = opcode & 0x7f
    position = 2
    if offset:
        last_index += offset
    else:
        if len(data) < 4:
            raise ValueError("absolute SetMap needs a tile index")
        last_index = int.from_bytes(data[2:4], "little")
        position = 4
    field_widths = ((1, 2), (2, 4), (4, 4), (8, 2), (16, 1), (32, 4), (64, 6), (128, 1))
    for flag, width in field_widths:
        if flags & flag:
            position += width
    return position, last_index

def split_payload(payload):
    position = 0
    last_index = 0
    commands = []
    while position < len(payload):
        opcode = payload[position]
        remaining = payload[position:]
        if opcode & 128:
            length, last_index = setmap_length(remaining, last_index)
        else:
            length = FIXED_LENGTHS.get(opcode, 16)
        if length <= 0 or position + length > len(payload):
            raise ValueError(f"truncated opcode {opcode} at {position}: need {length}, have {len(payload) - position}")
        commands.append(payload[position:position + length])
        position += length
    return commands

for message in fixtures:
    parsed = split_payload(message.data)
    assert len(parsed) == 1 and len(parsed[0]) == len(message.data), message

assert len(split_payload(fixtures[0].data + fixtures[2].data)) == 2
try:
    split_payload(bytes([128, 1]))
except ValueError:
    pass
else:
    raise AssertionError("truncated SetMap was accepted")

print(f"Validated {len(fixtures)} protocol fixtures and truncation handling.")

Validated 58 protocol fixtures and truncation handling.


## Persistent compression harness

`PersistentStream` mirrors the server's per-player encoder and the client's continuous decoder. The emitted compressed bytes are measured as the new bytes produced by one sync-flushed batch, which is the same delta concept used by `compress_ticks()`.

In [4]:
class PersistentStream:
    def __init__(self, level=9):
        self.encoder = zlib.compressobj(level=level, wbits=zlib.MAX_WBITS)
        self.decoder = zlib.decompressobj(wbits=zlib.MAX_WBITS)

    def compress_segment(self, payload):
        emitted = self.encoder.compress(payload)
        emitted += self.encoder.flush(zlib.Z_SYNC_FLUSH)
        recovered = self.decoder.decompress(emitted)
        assert recovered == payload, "persistent zlib round-trip failed"
        return emitted

    def measure_batch(self, payload):
        raw_payload = len(payload)
        raw_wire = raw_payload + 2
        compressed_payload = len(self.compress_segment(payload))
        compressed_wire = compressed_payload + 2
        selected = raw_wire > COMPRESSION_THRESHOLD
        selected_wire = compressed_wire if selected else raw_wire
        return {
            "raw_payload": raw_payload,
            "raw_wire": raw_wire,
            "compressed_payload": compressed_payload,
            "compressed_wire": compressed_wire,
            "selected": selected,
            "selected_wire": selected_wire,
            "savings_bytes": raw_wire - selected_wire,
        }

def batch(messages):
    return b"".join(message.data for message in messages)

boundary_stream = PersistentStream()
boundary_results = [boundary_stream.measure_batch(bytes(size)) for size in (0, 2, 14, 15, 16, 32)]
assert [result["selected"] for result in boundary_results] == [False, False, False, True, True, True]

print("Boundary check passed: raw payload 15 bytes is the first compressed case because its framed length is 17.")

Boundary check passed: raw payload 15 bytes is the first compressed case because its framed length is 17.


In [15]:
by_family = defaultdict(list)
for message in fixtures:
    by_family[message.family].append(message)

def pick(family, variant=None):
    choices = by_family[family]
    if variant is None:
        return choices[0]
    return next(message for message in choices if message.variant == variant)

def choose(family, index):
    choices = by_family[family]
    return choices[index % len(choices)]

def dynamic_bytes(seed, count):
    rng = random.Random(SEED + seed * 7919)
    return bytes(rng.randrange(256) for _ in range(count))

def dynamic_u16_bytes(seed, count):
    rng = random.Random(SEED + seed * 7919)
    return b"".join(le16(rng.randrange(65536)) for _ in range(count))

def dynamic_tick(tick):
    return make("Tick", "dynamic", fixed(27, bytes([tick & 0xff])))

def dynamic_map_messages(tick):
    messages = [dynamic_tick(tick)]
    for offset in range(8):
        delta = 4 if offset % 2 == 0 else 7
        flags = 1 if offset % 2 == 0 else 255
        seed = 21 + tick * 8 + offset
        messages.append(set_map("dynamic", delta, flags, 2052 + tick * 8 + offset, seed))
    messages.append(make(
        "SetMap3",
        "dynamic",
        fixed(45, le16(1200 + tick) + bytes([8]) + bytes(((0x87 + tick + index) & 0xff) for index in range(13))),
    ))
    messages.append(make(
        "SetMap6",
        "dynamic",
        fixed(68, le16(1200 + tick) + bytes([8, (0x87 + tick) & 0xff, (0x65 + tick) & 0xff, (0x43 + tick) & 0xff])),
    ))
    return messages

def dynamic_movement_messages(tick):
    scroll_opcodes = (30, 31, 32, 33, 35, 36, 37, 38)
    map_delta = 4 if tick % 2 == 0 else 7
    map_flags = 1 if tick % 2 == 0 else 255
    return [
        dynamic_tick(tick),
        make("Scroll", "dynamic", bytes([scroll_opcodes[tick % len(scroll_opcodes)]])),
        make("SetOrigin", "dynamic", fixed(44, le16(80 + tick * 3) + le16(120 + tick * 2))),
        set_map("dynamic", map_delta, map_flags, 1200 + tick * 4, 40 + tick),
        make("SetMap4", "dynamic", fixed(66, le16(1200 + tick * 4) + bytes([(tick * 13) & 0xff]))),
    ]

def dynamic_character_state_messages(tick):
    messages = [
        dynamic_tick(tick),
        make("SetCharMode", "dynamic", fixed(6, bytes([tick % 8]))),
        make("SetCharAttrib", "dynamic", fixed(7, bytes([tick % 6]) + dynamic_u16_bytes(tick + 100, 6))),
        make("SetCharSkill", "dynamic", fixed(8, bytes([tick % 12]) + dynamic_u16_bytes(tick + 200, 6))),
    ]
    for opcode, family, seed in [
        (12, "SetCharHp", 300),
        (13, "SetCharEndur", 400),
        (14, "SetCharMana", 500),
    ]:
        messages.append(make(family, "dynamic", fixed(opcode, dynamic_u16_bytes(tick + seed, 6))))
    for opcode, family, seed in [
        (20, "SetCharAHP", 600),
        (69, "SetCharAEnd", 700),
        (70, "SetCharAMana", 800),
    ]:
        messages.append(make(family, "dynamic", fixed(opcode, le16(2 + ((tick + seed) % 1000)))))
    messages.extend([
        make("SetCharPts", "dynamic", fixed(21, le16(tick % 100) + le32(10000 + tick * 17) + bytes([tick % 8]) + dynamic_bytes(tick + 900, 5))),
        make("SetCharGold", "dynamic", fixed(22, le32(8000 + tick * 31) + le16(101 + tick) + le16(202 + tick * 2) + dynamic_bytes(tick + 1000, 4))),
        make("SetCharItem", "dynamic", fixed(23, le16(7 + tick % 16) + le16(320 + tick) + le32(123456 + tick * 97))),
        make("SetCharWorn", "dynamic", fixed(24, le16(4 + tick % 8) + le16(900 + tick) + le32(654321 + tick * 113))),
        make("SetCharSpell", "dynamic", fixed(46, le16(tick % 12) + dynamic_bytes(tick + 1100, 8))),
        make("SetCharObj", "dynamic", fixed(25, le16(16 + tick % 32) + le16(700 + tick * 5))),
    ])
    return messages

def dynamic_login_snapshot_messages(tick):
    return [
        dynamic_tick(tick),
        make("SetCharName1", "dynamic", fixed16(3, f"Player{tick:09d}".encode("ascii"))),
        make("SetCharTalents", "dynamic", fixed(75, dynamic_bytes(tick + 1200, 25))),
        make("SetCharRuneState", "dynamic", fixed(78, bytes([tick % 8]) + le16((tick * 37) & 0xffff))),
        make("SetCompletionData", "dynamic", fixed(77, dynamic_bytes(tick + 1300, 41))),
        make("SetWeather", "dynamic", fixed(76, bytes([tick % 5, 180 + tick % 70]) + le16(720 + tick * 3) + dynamic_bytes(tick + 1400, 5))),
        make("SetOrigin", "dynamic", fixed(44, le16(80 + tick * 3) + le16(120 + tick * 2))),
    ]

def dynamic_log_message(tick):
    return make("Log", "dynamic", fixed16(52, f"{tick % 100000:05d} says hello".encode("ascii")))

def dynamic_communication_messages(tick):
    return [
        dynamic_tick(tick),
        dynamic_log_message(tick),
        make("Look", "dynamic", fixed16(29, dynamic_bytes(tick + 1500, 15))),
        make("SetTarget", "dynamic", fixed(42, le16(900 + tick) + le16(81 + tick * 2) + le16(90 + tick * 3) + bytes([3, 4, 5]) + dynamic_bytes(tick + 1600, 3))),
        make("PlaySound", "dynamic", fixed(47, le16(12 + tick % 20) + bytes([90 + tick % 20, 128]) + dynamic_bytes(tick + 1700, 8))),
        make("Pong", "dynamic", fixed16(74, le32(123456 + tick * 1000) + le32(7890 + tick) + dynamic_bytes(tick + 1800, 7))),
    ]

def workload_messages(name, tick):
    if name == "idle":
        return [dynamic_tick(tick)]
    if name == "movement":
        return dynamic_movement_messages(tick)
    if name == "map-heavy":
        return dynamic_map_messages(tick)
    if name == "character-state":
        return dynamic_character_state_messages(tick)
    if name == "login-snapshot":
        return dynamic_login_snapshot_messages(tick)
    if name == "communication":
        return dynamic_communication_messages(tick)
    if name == "normal-mixed":
        return dynamic_movement_messages(tick) + dynamic_character_state_messages(tick)[:4] + [dynamic_log_message(tick)]
    raise KeyError(name)

WORKLOADS = ["idle", "movement", "map-heavy", "character-state", "login-snapshot", "communication", "normal-mixed"]
for workload in WORKLOADS:
    assert workload_messages(workload, 0), workload

for workload in WORKLOADS:
    payloads = [batch(workload_messages(workload, tick)) for tick in range(8)]
    assert len(set(payloads)) == len(payloads), f"{workload} fixtures repeat within the check window"
    for tick, payload in enumerate(payloads):
        messages = workload_messages(workload, tick)
        assert len(split_payload(payload)) == len(messages), f"invalid {workload} payload at tick {tick}"

print({workload: sum(len(message.data) for message in workload_messages(workload, 0)) for workload in WORKLOADS})

{'idle': 2, 'movement': 16, 'map-heavy': 146, 'character-state': 140, 'login-snapshot': 105, 'communication': 76, 'normal-mixed': 64}


## Primary table 1: average compression benefit by message type

A message is measured after a shared prefix has already established a zlib dictionary. This is a **marginal contribution**, not an intrinsic compression ratio: neighboring messages influence the result. Each sample uses a sync-flushed segment so the marginal compressed bytes are observable, while the two-byte frame header is deliberately excluded from per-message attribution because production adds one header per batch, not per message.

In [16]:
context_prefixes = [
    [pick("Tick", "idle"), pick("SetOrigin")],
    workload_messages("movement", 1),
    workload_messages("character-state", 2)[:5],
    workload_messages("normal-mixed", 3),
]

marginal_rows = []
for message in fixtures:
    for prefix in context_prefixes:
        stream = PersistentStream()
        stream.compress_segment(batch(prefix))
        emitted = stream.compress_segment(message.data)
        raw_bytes = len(message.data)
        compressed_bytes = len(emitted)
        marginal_rows.append({
            "message_type": message.family,
            "raw_bytes": raw_bytes,
            "compressed_bytes": compressed_bytes,
            "savings_bytes": raw_bytes - compressed_bytes,
            "savings_pct": 100 * (raw_bytes - compressed_bytes) / raw_bytes,
        })

benefit_groups = defaultdict(list)
for row in marginal_rows:
    benefit_groups[row["message_type"]].append(row)
benefit_table = []
for family, rows in benefit_groups.items():
    benefit_table.append({
        "message_type": family,
        "samples": len(rows),
        "avg_raw_B": statistics.mean(row["raw_bytes"] for row in rows),
        "avg_compressed_B": statistics.mean(row["compressed_bytes"] for row in rows),
        "avg_savings_B": statistics.mean(row["savings_bytes"] for row in rows),
        "avg_savings_pct": statistics.mean(row["savings_pct"] for row in rows),
    })
benefit_table.sort(key=lambda row: row["avg_savings_pct"], reverse=True)

def print_table(rows, columns, headers=None):
    headers = headers or columns
    header_values = [str(header) for header in headers]
    table_values = [header_values] + [
        [str(row[column]) for column in columns]
        for row in rows
    ]
    widths = [max(len(values[index]) for values in table_values) for index in range(len(columns))]

    def format_row(values):
        padded = [value.ljust(width) for value, width in zip(values, widths)]
        return "| " + " | ".join(padded) + " |"

    separator = "+-" + "-+-".join("-" * width for width in widths) + "-+"
    print(separator)
    print(format_row(header_values))
    print(separator)
    for values in table_values[1:]:
        print(format_row(values))
    print(separator)

def rounded_rows(rows, columns, digits=2):
    result = []
    for row in rows:
        copy = dict(row)
        for column in columns:
            if isinstance(copy.get(column), float):
                copy[column] = round(copy[column], digits)
        result.append(copy)
    return result

benefit_display = rounded_rows(benefit_table, ["avg_raw_B", "avg_compressed_B", "avg_savings_B", "avg_savings_pct"])
print_table(benefit_display, ["message_type", "samples", "avg_raw_B", "avg_compressed_B", "avg_savings_B", "avg_savings_pct"], ["Message type", "Samples", "Raw B", "Marginal compressed B", "Savings B", "Savings %"])

+-------------------+---------+-------+-----------------------+-----------+-----------+
| Message type      | Samples | Raw B | Marginal compressed B | Savings B | Savings % |
+-------------------+---------+-------+-----------------------+-----------+-----------+
| SetCompletionData | 8       | 42    | 31                    | 11        | 26.19     |
| SetMap3           | 4       | 17    | 13                    | 4         | 23.53     |
| SetCharTalents    | 8       | 26    | 21.5                  | 4.5       | 17.31     |
| Pong              | 4       | 16    | 16                    | 0         | 0.0       |
| PlaySound         | 4       | 13    | 14                    | -1        | -7.69     |
| Look              | 8       | 16    | 17.5                  | -1.5      | -9.38     |
| SetCharName1      | 8       | 16    | 18.5                  | -2.5      | -15.62    |
| SetCharAttrib     | 12      | 14    | 17                    | -3        | -21.43    |
| SetWeather        | 8       | 

## Primary table 2: bandwidth with and without compression

This table models each workload over a warmed persistent stream. `Raw wire B/tick` includes the 2-byte header for the uncompressed path. `Selected wire B/tick` follows the current server decision exactly: compression is used above the threshold even if a particular batch's compressed representation is not smaller.

In [17]:
def measure_workload(workload, ticks=BENCHMARK_TICKS, warmup=WARMUP_TICKS):
    stream = PersistentStream()
    for tick in range(warmup):
        stream.measure_batch(batch(workload_messages(workload, tick)))
    records = []
    for tick in range(ticks):
        messages = workload_messages(workload, tick + warmup)
        payload = batch(messages)
        record = stream.measure_batch(payload)
        record["workload"] = workload
        record["message_count"] = len(messages)
        records.append(record)
    return records

scenario_records = {workload: measure_workload(workload) for workload in WORKLOADS}
bandwidth_table = []
for workload in WORKLOADS:
    rows = scenario_records[workload]
    raw_wire = statistics.mean(row["raw_wire"] for row in rows)
    compressed_wire = statistics.mean(row["compressed_wire"] for row in rows)
    selected_wire = statistics.mean(row["selected_wire"] for row in rows)
    raw_per_second = raw_wire * TICKS_PER_SECOND
    selected_per_second = selected_wire * TICKS_PER_SECOND
    bandwidth_table.append({
        "workload": workload,
        "raw_payload_B_tick": statistics.mean(row["raw_payload"] for row in rows),
        "raw_wire_B_tick": raw_wire,
        "compressed_wire_B_tick": compressed_wire,
        "selected_wire_B_tick": selected_wire,
        "selected_savings_B_tick": raw_wire - selected_wire,
        "selected_savings_pct": 100 * (raw_wire - selected_wire) / raw_wire if raw_wire else 0,
        "raw_B_s_player": raw_per_second,
        "selected_B_s_player": selected_per_second,
        "selected_Mbps": selected_per_second * ACTIVE_PLAYERS * 8 / 1_000_000,
        "compressed_selected_pct": 100 * statistics.mean(row["selected"] for row in rows),
    })

bandwidth_display = rounded_rows(bandwidth_table, ["raw_payload_B_tick", "raw_wire_B_tick", "compressed_wire_B_tick", "selected_wire_B_tick", "selected_savings_B_tick", "selected_savings_pct", "raw_B_s_player", "selected_B_s_player", "selected_Mbps", "compressed_selected_pct"])
print_table(bandwidth_display, ["workload", "raw_payload_B_tick", "raw_wire_B_tick", "selected_wire_B_tick", "selected_savings_B_tick", "selected_savings_pct", "selected_B_s_player", "selected_Mbps", "compressed_selected_pct"], ["Workload", "Raw payload B/tick", "Raw wire B/tick", "Selected wire B/tick", "Savings B/tick", "Savings %", "Selected B/s/player", f"Selected Mbps ({ACTIVE_PLAYERS} players)", "Compressed frames %"])

+-----------------+--------------------+-----------------+----------------------+----------------+-----------+---------------------+-----------------------------+---------------------+
| Workload        | Raw payload B/tick | Raw wire B/tick | Selected wire B/tick | Savings B/tick | Savings % | Selected B/s/player | Selected Mbps (100 players) | Compressed frames % |
+-----------------+--------------------+-----------------+----------------------+----------------+-----------+---------------------+-----------------------------+---------------------+
| idle            | 2                  | 4               | 4                    | 0              | 0.0       | 144                 | 0.12                        | 0                   |
| movement        | 27                 | 29              | 32.03                | -3.03          | -10.45    | 1153.08             | 0.92                        | 100                 |
| map-heavy       | 146                | 148             | 100.52          

In [18]:
best = max(bandwidth_table, key=lambda row: row["selected_savings_pct"])
worst = min(bandwidth_table, key=lambda row: row["selected_savings_pct"])
print(f"At {ACTIVE_PLAYERS} active players and {TICKS_PER_SECOND} ticks/s, the largest synthetic saving is {best['workload']} at {best['selected_savings_pct']:.1f}% ({best['selected_Mbps']:.3f} Mbps after selection).")
print(f"The weakest case is {worst['workload']} at {worst['selected_savings_pct']:.1f}%; idle or below-threshold batches retain the raw framed size.")
print("Interpretation: table 1 is useful for comparing message families, while table 2 is the decision table because compression happens on complete tick batches and benefits depend on message order and stream history.")

At 100 active players and 36 ticks/s, the largest synthetic saving is map-heavy at 32.1% (2.895 Mbps after selection).
The weakest case is movement at -10.4%; idle or below-threshold batches retain the raw framed size.
Interpretation: table 1 is useful for comparing message families, while table 2 is the decision table because compression happens on complete tick batches and benefits depend on message order and stream history.


## Limitations

- Fixtures are deterministic and synthetic; they do not replace a captured `tbuf` replay corpus.
- Workload generators use evolving protocol-shaped values to avoid overstating dictionary reuse from repeated fixture cycles.
- Marginal message attribution is context-dependent because zlib shares a dictionary across a batch and across frames.
- Python timing does not measure Rust `flate2` CPU cost.
- The current server threshold is reproduced, including cases where compression could be larger than raw framing.
- `csend()` traffic such as login acknowledgements is outside the main result.
- Exact byte parity should be spot-checked with a Rust `flate2` probe if these values become an operational capacity commitment.

## Observed server data

The info-level capture in `server/logs/server.log` contains one player session with 1,093 tick records, covering ticks `1517775340` through `1517776432` (about 30 seconds at 36 ticks/second). The calculations below use `raw_framed = raw_len + 2` and compare it with the logged `wire_len`.

```text
Phase                              Records  Raw framed B  Wire B  Savings B  Savings %
All logged frames                  1093     145769        23425   122344     83.93
Non-empty frames (`raw_len > 0`)   96       143775        21431   122344     85.09
Compressed frames                  65       143410        21066   122344     85.31
Compressed after initial snapshot  64       93536         13001   80535      86.10
```

The first record is likely the initial login/world snapshot: `49872 B` raw framed data became `8065 B`, saving `41809 B` or `83.83%`. The remaining capture contains 997 empty frames, 31 non-empty frames below the compression threshold, and 64 further compressed frames.

Only 65 of 1,093 records were compressed, but they carried about 99.8% of the raw payload bytes. This means the bandwidth benefit comes from large map/state bursts rather than ordinary small tick frames. The capture averages about `4.8 KB/s` of raw framed data versus `0.77 KB/s` on the wire for this player.

Two compressed frames were larger than their raw framed equivalents, matching the server's current threshold-based selection policy:

- Tick `1517775740`: `50 B` raw framed to `60 B` on wire (`-10 B`).
- Tick `1517776343`: `34 B` raw framed to `40 B` on wire (`-6 B`).

This is one player/session, and compressed `payload_len` values are the incremental bytes emitted by the persistent zlib stream after a sync flush, not standalone compressed-stream sizes. The capture strongly supports compression for large gameplay bursts, but more sessions and longer steady-state captures are needed for a capacity estimate.